# WSJ 2027 - Gruppindelning Rundresa

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort, friend-cluster-aware)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Kår-konsolation** — alla med en kår (≥2 medlemmar på resan) ska helst ha en kår-kompis i gruppen, om de inte fick någon vän-önskan satt. Gäller även dem som inte lämnat något vän-önskemål (soft goal; Phase 3.5 + Phase 4 SA)
5. **Max 6 from same kår** per group (hard constraint)
6. **Diversity** — age (14-17), sex och samverkansorganisation should be as evenly spread as possible


In [1]:
TRAVEL = 'rundresa'        # only line that differs between rundresa and direktresa
QUALITY = 'slow'           # 'medium' or 'slow' (8 restarts)
WEIGHT_PROFILE = 'friend_geo'  # 'balanced' or 'friend_geo'
COORD_SOURCE = 'home'      # 'kar' (kår-first) or 'home' (home-address-first)
GROUP_SIZE = 36
MAX_KAR = 6
OUTPUT_DIR = '/config/notebooks/wsj27/output'


In [2]:
import sys; sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

raw = u.fetch_participants()
df_all, _ = u.build_participant_dataframe(raw, include_reserves=True)
df = df_all[(df_all['travel'] == TRAVEL) & (df_all['category'] == 'deltagare')].copy().reset_index(drop=True)
df = u.select_top_reserves(df, group_size=GROUP_SIZE, df_full=df_all).reset_index(drop=True)
u.assign_coordinates(df, coord_source=COORD_SOURCE)
df = u.add_hilbert_index(df)
u.resolve_friend_wishes(df, df_all)
u.apply_manual_overrides(df, df_all)
friend_wishes = u.build_friend_graph(df)
u.print_intake_summary(df, GROUP_SIZE)


Fetched 2698 participants
Confirmed: 2613, Unconfirmed: 6, Cancelled: 79
Loaded 314 samverkansorganisation entries from World Scout Jamboree - Polen 2027, Deltagare _ IST (Funktionar) - deltakere (2).xlsx
Total participants: 2610 (2561 confirmed, 49 reservlista)
Skipped: 6 unconfirmed, 0 nekad (reservlista included), 3 wrong age/no DOB

By category:
category
deltagare    1927
ist           645
cmt            38

By travel type:
travel
rundresa      1502
direktresa     569
egen_resa      297
other          242

Friend wishes:
  With friend 1 member no: 1357
  With friend 2 member no: 805
  With friend 1 name (text): 118
  With friend 2 name (text): 80

Skipped participants:
  DELTAGARE wrong age: Adam Egelnor born 2009-05-25 (age 18)
  DELTAGARE wrong age: Attle Käck Karlsson born 2004-03-09 (age 23)
  DELTAGARE wrong age: Elisabeth Undén born 1976-11-21 (age 50)
(confirmed cohort 1332 already fills groups; skipping reserves)
  (rejected 16 home-address geocodes outside Sweden bbox; wil

In [3]:
df = u.assign_groups(df, GROUP_SIZE, friend_wishes,
                     max_kar=MAX_KAR, quality=QUALITY,
                     weight_profile=WEIGHT_PROFILE)
u.apply_manual_placement(df)
group_of = df['group'].values
total_groups = df['group'].nunique()



############################################################
# Slow tier: 8 restarts
############################################################

----- Restart 1/8 (seed=42) -----
Participants: 1332
Groups: 37 x 36 + 1 x 0 = 37 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 929/929
  Kar violations: 162
  Avg geo spread: 2.3661

=== Phase 2: Fix friend wishes (criticality-ordered, iterate) ===
  No unsatisfied wishes; converged after 0 pass(es)
  Total swaps: 0
  Friend satisfaction: 929/929
  Kar violations: 162
  Avg geo spread: 2.3661

=== Phase 2.5: 3-way rotations ===
  Rotations: 0
  Friend satisfaction: 929/929
  Kar violations: 162

=== Phase 3: Fix kar violations (friend-aware) ===
  Swaps: 158
  Kar violations: 0
  Friend satisfaction: 827/929
  Avg geo spread: 2.4242

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 74
  Friend satisfaction: 900/929
  Kar violations: 0
  Avg geo spread: 2.4569

=== Phase 3.5: Kår-konsolation (targeted) ===
  Sw

In [4]:
u.print_group_metrics(df, group_of, total_groups)
csv_path, json_path = u.export_results(df, group_of, total_groups, OUTPUT_DIR, prefix=f'wsj27_{TRAVEL}')
map_path = f'{OUTPUT_DIR}/wsj_{TRAVEL}_karta.html'
u.generate_group_map_html(df, total_groups, map_path,
                          title=f'WSJ 2027 {TRAVEL.title()}',
                          friend_wishes=friend_wishes, show_group_arcs=True)
report_path = f'{OUTPUT_DIR}/wsj_{TRAVEL}_grupper.html'
u.generate_groups_report_html(df, total_groups, report_path,
                              title=f'WSJ 2027 {TRAVEL.title()} - Grupper',
                              friend_wishes=friend_wishes,
                              group_size=GROUP_SIZE,
                              max_kar=MAX_KAR,
                              quality=QUALITY,
                              weight_profile=WEIGHT_PROFILE,
                              coord_source=COORD_SOURCE)
print(f'CSV:    {csv_path}\nJSON:   {json_path}\nMap:    {map_path}\nReport: {report_path}')


Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 12/13      5   16/20/0  10  10  10   6       214    19
        Orter: Dalby (5), Veberöd (4), Svedala (3), Lomma (3), Trelleborgs (2), Staffanstorps Scoutkår Torparna (2), Torslanda (2), Wasakåren (2), Equmenia Storvreta (2), Knivsta (2), Södra Sandby (1), Örtofta (1), Tor (1), Hovsta (1), Landeryd (1), Equmenia Scout (1), Sofia (1), Glimåkra (1), Östersund Frösö (1)
    2      36 24/25      5   17/19/0   6   7  11  12       148    13
        Orter: Dalby (5), Torns (5), Finn (5), Eslövs (4), Rydebäcks (3), Södra Sandby (2), Furulunds (2), Svedala (2), Karlsro (2), Almtuna (2), Wasakåren (2), Höörs (1), Dösjebro (1)
    3      36 29/30      5   20/16/0   7   7  13   9        16    14
        Orter: Dalby (5), Furulunds (4), Finn (4), Staffanstorps Scoutkår Torparna (4), Vellinge (3), Tornugglan (3), Landskrona (2), Månstorp